In [1]:
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

from DL.Lab8.CustomSpeachDataset import CustomSpeachDataset
from DL.Lab8.SpeachClassifierModel import SpeachClassifierModel

2026-05-21 17:44:41.787478: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [3]:
MODELS_DIR = Path("checkpoints")
MODELS_DIR.mkdir(exist_ok=True)

In [5]:
dataset = CustomSpeachDataset(Path("../preprocessed_dataset"), preload=True)
dataset.to(device)

Preloading dataset from disk...


/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
train_val_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_val_indices]
train_val_dataset = torch.utils.data.Subset(dataset, train_val_indices)
train_val_sample_weights = all_sample_weights[train_val_indices]

test_dataset = torch.utils.data.Subset(dataset, test_indices)

len(train_val_dataset), len(y_train)

(14679, 14679)

In [7]:
dataset.y.unique()

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [8]:
def train_new_model(writer: SummaryWriter, fold: int, params: dict, epochs: int, train_loader: DataLoader, val_loader: DataLoader, save_dir: Path, training_state: dict|None) -> SpeachClassifierModel:
    model = SpeachClassifierModel(params["dropout_rate"])
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    best_val_loss = 99999999999.9

    loss_fn = nn.CrossEntropyLoss()

    start_epoch = 0

    if training_state:
        optimizer.load_state_dict(training_state["optimizer"])
        best_val_loss = training_state["best_val_loss"]
        start_epoch = training_state["epoch"]

    for epoch in range(start_epoch, epochs):
        print(f"Fold {fold} starting epoch {epoch}")

        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()

            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                y_pred = model(data)
                loss = loss_fn(y_pred, target)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_path = save_dir / "best" / f"fold{fold}.pth"
            torch.save(model.state_dict(), save_path)

        writer.add_scalar(f"Loss/train/fold{fold}", train_loss, epoch)
        writer.add_scalar(f"Loss/val/fold{fold}", val_loss, epoch)

        torch.save(model.state_dict(), save_dir / "latest" / f"fold{fold}.pth")

        training_state = {
            "epoch": epoch,
            "optimizer": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "fold": fold,
        }
        torch.save(training_state, save_dir / "latest" / f"training_state.pth")

    model.best_val_loss = best_val_loss

    return model

In [9]:
params = {
    "lr": 0.001,
    "dropout_rate": 0.15,
    "weight_decay": 0.0006,
}
save_dir = MODELS_DIR / "training_base"
save_dir.mkdir(parents=True, exist_ok=True)

(save_dir/"best").mkdir(parents=True, exist_ok=True)
(save_dir/"latest").mkdir(parents=True, exist_ok=True)
torch.save(params, save_dir / "best" / "model_params.pth")
torch.save(params, save_dir / "latest" / "model_params.pth")

epochs = 100

train_sampler = WeightedRandomSampler(
    weights=train_val_sample_weights,
    num_samples=len(train_val_sample_weights),
    replacement=True,
)
train_loader = DataLoader(train_val_dataset, batch_size=32, sampler=train_sampler)
val_loader = DataLoader(test_dataset, batch_size=1024)

with SummaryWriter(f"runs/training/folds") as writer:
    avg_val_loss = train_new_model(writer, 9, params, epochs, train_loader, val_loader, save_dir, training_state=None)


Fold 9 starting epoch 0
Fold 9 starting epoch 1


KeyboardInterrupt: 